# GIOŚ — import danych do warstwy RAW

## Cel
Pobranie oryginalnych plików źródłowych GIOŚ i zapisanie ich lokalnie wraz z metadanymi.

Na tym etapie nie rozpakowujemy archiwów, nie czyścimy danych i nie tworzymy warstwy bronze.

Kod pobierania znajduje się w `scripts/download_raw.py`.

## 1. Instalacja zależności

Komórka instaluje zależności w środowisku aktualnie wybranego kernela.

In [ ]:
%pip install requests duckdb polars pyarrow scikit-learn

## 2. Lokalizacja projektu

Notebook można uruchomić z katalogu głównego projektu albo z katalogu `notebooks`.

In [ ]:
from pathlib import Path
import json
import runpy
import sys
import platform
from importlib.metadata import version

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
CONFIG_PATH = ROOT / 'config' / 'sources.json'
RAW_DIR = ROOT / 'data' / 'raw'
SCRIPT_PATH = ROOT / 'scripts' / 'download_raw.py'

if not CONFIG_PATH.is_file() or not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        'Uruchom notebook z katalogu projektu lub notebooks.'
    )

RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Projekt:', ROOT)
print('Interpreter:', sys.executable)
print('Python:', platform.python_version())

for package in [
    'requests', 'duckdb', 'polars', 'pyarrow', 'scikit-learn'
]:
    print(f'{package}: {version(package)}')

## 3. Konfiguracja źródeł

Przed importem uzupełnij `config/sources.json` bezpośrednimi adresami plików źródłowych.

Źródło danych: Główny Inspektorat Ochrony Środowiska.

Bank danych: https://powietrze.gios.gov.pl/pjp/archives

In [ ]:
config = json.loads(
    CONFIG_PATH.read_text(encoding='utf-8-sig')
)

print(json.dumps(config, ensure_ascii=False, indent=2))

## 4. Import do RAW

Uruchamiamy funkcję importu z modułu pomocniczego.

Dla każdego pliku zapisujemy oryginalną zawartość oraz metadane: URL, datę pobrania, rozmiar i SHA-256.

Poprawne, wcześniej pobrane pliki są pomijane.

In [ ]:
import_module = runpy.run_path(str(SCRIPT_PATH))
import_module['main']()

## 5. Podsumowanie lokalnych danych RAW

Zestawienie poniżej obejmuje pliki znajdujące się w warstwie RAW i posiadające plik metadanych. Nie oznacza liczby rekordów pomiarowych — ich liczenie będzie częścią kolejnego etapu.

In [ ]:
records = []

for metadata_path in sorted(
    RAW_DIR.rglob('*.metadata.json')
):
    metadata = json.loads(
        metadata_path.read_text(encoding='utf-8')
    )
    data_path = metadata_path.parent / metadata['filename']
    if data_path.is_file():
        records.append(metadata)

total_bytes = sum(item['size_bytes'] for item in records)

print('Liczba plików:', len(records))
print(f'Łączny rozmiar: {total_bytes / 1_000_000:.2f} MB')

for item in records:
    print()
    print('Plik:', item['filename'])
    print('Rok:', item['year'])
    print('Bajty:', item['size_bytes'])
    print('Pobrano UTC:', item['downloaded_at_utc'])
    print('SHA-256:', item['sha256'])

## 6. Wnioski z etapu RAW

Po wykonaniu importu uzupełnij:

- Jakie lata i pliki pobrano?
- Jaki jest rozmiar danych?
- Czy wystąpiły błędy pobierania?
- Jaki zakres danych wybrano do dalszej analizy?

Następny etap: odczyt plików źródłowych i przygotowanie warstwy bronze.